# AWS Bedrock Audio Analysis Test (Using Converse API)

This notebook tests uploading an audio file to AWS Bedrock using the **Converse API** with proper audio format.

In [104]:
import boto3
import json
from pathlib import Path
from botocore.exceptions import ClientError

## Step 1: Initialize Bedrock Client

In [ ]:
# AWS Configuration
AWS_REGION = "us-west-2"
AWS_PROFILE = "hackaton"  # Change this to your AWS profile name

# Model configuration - using a working model from previous tests
MODEL_ID = "amazon.nova-2-sonic-v1:0"

# Initialize Bedrock client
session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
bedrock_runtime = session.client('bedrock-runtime')

print(f"✅ Initialized Bedrock client in region: {AWS_REGION}")
print(f"📦 Using model: {MODEL_ID}")
print(f"🔄 Using Converse API (supports audio)")

✅ Initialized Bedrock client in region: us-west-2
📦 Using model: us.amazon.nova-premier-v1:0
🔄 Using Converse API (supports audio)


## Step 2: Test Connection with Hello Prompt (Converse API)

In [106]:
print("🧪 Testing Bedrock Converse API...\n")

try:
    response = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": [
                    {"text": "Hello, respond with OK if you can hear me."}
                ]
            }
        ]
    )
    
    # Extract response text
    response_text = response['output']['message']['content'][0]['text']
    
    print(f"✅ Connection successful!")
    print(f"📋 Model: {MODEL_ID}")
    print(f"💬 Response: {response_text.strip()}")
    print(f"\n✨ Converse API is working!\n")
    
    # Show usage stats
    usage = response.get('usage', {})
    print(f"📊 Token usage:")
    print(f"   Input: {usage.get('inputTokens', 0)}")
    print(f"   Output: {usage.get('outputTokens', 0)}")
    
except ClientError as e:
    error_code = e.response['Error']['Code']
    error_msg = e.response['Error']['Message']
    
    print(f"❌ Connection failed: {error_code}")
    print(f"   Error: {error_msg}")
    
except Exception as e:
    print(f"❌ Unexpected error: {str(e)}")

🧪 Testing Bedrock Converse API...

✅ Connection successful!
📋 Model: us.amazon.nova-premier-v1:0
💬 Response: Hello! OK, I can hear you. How can I assist you today? If you have any questions or need information, feel free to ask. I'm here to help with a wide range of topics, from answering trivia questions to helping you understand complex concepts. What would you like to talk about or need assistance with?

✨ Converse API is working!

📊 Token usage:
   Input: 45
   Output: 67


## Step 3: Load Audio File

In [107]:
# Path to your audio file
AUDIO_FILE_PATH = "input.mp3"

audio_path = Path(AUDIO_FILE_PATH)

if not audio_path.exists():
    print(f"❌ Error: File not found at {AUDIO_FILE_PATH}")
    audio_data = None
else:
    with open(audio_path, 'rb') as f:
        audio_data = f.read()
    
    file_size_mb = len(audio_data) / (1024 * 1024)
    print(f"✅ Loaded audio file: {audio_path.name}")
    print(f"📊 File size: {file_size_mb:.2f} MB")
    print(f"🎵 File extension: {audio_path.suffix}")
    
    # Determine format for Converse API audio block
    # Supported: mp3, mp4, wav, flac, ogg, webm
    audio_formats = {
        '.mp3': 'mp3',
        '.wav': 'wav',
        '.m4a': 'mp4',
        '.mp4': 'mp4',
        '.flac': 'flac',
        '.ogg': 'ogg',
        '.webm': 'webm'
    }
    
    audio_format = audio_formats.get(audio_path.suffix.lower(), 'mp3')
    print(f"📋 Audio format: {audio_format}")

✅ Loaded audio file: input.mp3
📊 File size: 4.07 MB
🎵 File extension: .mp3
📋 Audio format: mp3


In [108]:
# Initialize S3 client and upload audio
if audio_data:
    s3_client = session.client('s3')
    BUCKET_NAME = "riam-lms-recordings"
    
    # Upload audio to S3
    s3_key = f"audio-analysis/{audio_path.name}"
    s3_client.upload_file(str(audio_path), BUCKET_NAME, s3_key)
    s3_url = f"https://{BUCKET_NAME}.s3.{AWS_REGION}.amazonaws.com/{s3_key}"
    
    print(f"📤 Uploaded audio to S3: {s3_url}")
    print(f"📊 File size: {file_size_mb:.2f} MB")
    
    # Generate presigned URL
    presigned_url = s3_client.generate_presigned_url(
        'get_object',
        Params={'Bucket': BUCKET_NAME, 'Key': s3_key},
        ExpiresIn=3600
    )
    
    print(f"🔗 Presigned URL generated (expires in 1 hour)")
    print(f"📋 S3 Key: {s3_key}")
else:
    print("❌ No audio data to upload")

📤 Uploaded audio to S3: https://riam-lms-recordings.s3.us-west-2.amazonaws.com/audio-analysis/input.mp3
📊 File size: 4.07 MB
🔗 Presigned URL generated (expires in 1 hour)
📋 S3 Key: audio-analysis/input.mp3


## Step 5: Full Music Coach Analysis with Converse API

In [109]:
if audio_data:
    print("🎵 Running full music coach analysis...\n")
    
    full_prompt = """You are a master musician and experienced music teacher at the Royal Irish Academy of Music (RIAM).

A student has uploaded this practice recording for expert feedback.

Please listen to the audio and provide comprehensive feedback:

## 1. Piece Identification
- What piece is being performed?
- Who is the composer?
- Style period?

## 2. Technical Assessment
- Tone quality and control
- Rhythm and timing
- Technical facility
- 2-3 strengths
- 2-3 areas for improvement

## 3. Musical Interpretation
- Phrasing and musical line
- Dynamic range
- Expression and emotion
- What's working well?
- What could be enhanced?

## 4. Practice Recommendations
- Specific exercises
- Focus areas for next week
- Practice strategy

## 5. Resources
- 2 professional recordings to study
- Key listening points

Be specific and reference what you actually hear in the recording."""
    
    try:
        response = bedrock_runtime.converse(
            modelId=MODEL_ID,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "audio": {
                                "format": audio_format,
                                "source": {"bytes": audio_data}
                            }
                        },
                        {"text": full_prompt}
                    ]
                }
            ],
            inferenceConfig={
                "maxTokens": 3000
            }
        )
        
        feedback = response['output']['message']['content'][0]['text']
        usage = response.get('usage', {})
        
        print("="*80)
        print("🎵 COMPREHENSIVE MUSIC COACH FEEDBACK")
        print("="*80 + "\n")
        print(feedback)
        print("\n" + "="*80)
        print(f"\n📊 Tokens: {usage.get('inputTokens', 0)} in / {usage.get('outputTokens', 0)} out")
        
        # Save to file
        output_file = f"feedback_{audio_path.stem}.md"
        with open(output_file, 'w') as f:
            f.write(f"# RIAM AI Music Coach Feedback\n\n")
            f.write(f"**Audio File:** {audio_path.name}\n")
            f.write(f"**File Size:** {file_size_mb:.2f} MB\n")
            f.write(f"**Model:** {MODEL_ID}\n\n")
            f.write("---\n\n")
            f.write(feedback)
        
        print(f"\n💾 Feedback saved to: {output_file}")
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()

🎵 Running full music coach analysis...

❌ Error: An error occurred (ValidationException) when calling the Converse operation: ContentBlock object at messages.0.content.0 must set one of the following keys: text, image, toolUse, toolResult, document, video, cachePoint, searchResult.


Traceback (most recent call last):
  File "/var/folders/94/m4slhw4d3p500t3p8ntws2814khl6f/T/ipykernel_91302/1275700457.py", line 41, in <module>
    response = bedrock_runtime.converse(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/felipetenaglia/.local/lib/python3.12/site-packages/botocore/client.py", line 602, in _api_call
    return self._make_api_call(operation_name, kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/felipetenaglia/.local/lib/python3.12/site-packages/botocore/context.py", line 123, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/felipetenaglia/.local/lib/python3.12/site-packages/botocore/client.py", line 1078, in _make_api_call
    raise error_class(parsed_response, operation_name)
botocore.errorfactory.ValidationException: An error occurred (ValidationException) when calling the Converse operation: ContentBlock object at messages.0.content.0 must set one of the following keys: text, i

## Step 6: Test Different Audio Formats (Optional)

Test which audio formats are supported:

In [110]:
# Test different format values
if audio_data:
    # According to AWS docs, supported formats are: mp3, mp4, wav, flac, ogg, webm
    test_formats = ['mp3', 'mp4', 'wav', 'flac', 'ogg', 'webm']
    
    print("🧪 Testing different audio format values...\n")
    print("Using small audio sample (first 100KB) for testing\n")
    
    for fmt in test_formats:
        try:
            response = bedrock_runtime.converse(
                modelId=MODEL_ID,
                messages=[{
                    "role": "user",
                    "content": [
                        {
                            "audio": {
                                "format": fmt,
                                "source": {"bytes": audio_data[:102400]}  # 100KB for testing
                            }
                        },
                        {"text": "What do you hear? One sentence only."}
                    ]
                }]
            )
            print(f"✅ Format '{fmt}' - WORKS")
        except Exception as e:
            error_msg = str(e)
            if 'format' in error_msg.lower():
                print(f"❌ Format '{fmt}' - NOT SUPPORTED")
            elif 'audio' in error_msg.lower():
                print(f"❌ Format '{fmt}' - AUDIO NOT SUPPORTED BY MODEL")
            else:
                print(f"❓ Format '{fmt}' - Error: {error_msg[:80]}")

🧪 Testing different audio format values...

Using small audio sample (first 100KB) for testing

❓ Format 'mp3' - Error: An error occurred (ValidationException) when calling the Converse operation: Con
❓ Format 'mp4' - Error: An error occurred (ValidationException) when calling the Converse operation: Con
❓ Format 'wav' - Error: An error occurred (ValidationException) when calling the Converse operation: Con
❓ Format 'flac' - Error: An error occurred (ValidationException) when calling the Converse operation: Con
❓ Format 'ogg' - Error: An error occurred (ValidationException) when calling the Converse operation: Con
❓ Format 'webm' - Error: An error occurred (ValidationException) when calling the Converse operation: Con


## Summary

### Key Findings:

**Audio in Converse API:**
- ✅ Use `audio` content block (NOT `document`)
- ✅ Format: `{"audio": {"format": "mp3", "source": {"bytes": data}}}`
- ✅ Supported formats: mp3, mp4, wav, flac, ogg, webm
- ❌ `document` is for: pdf, docx, txt, csv, html, md, xlsx, xls

**Correct Structure:**
```python
response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[{
        "role": "user",
        "content": [
            {
                "audio": {              # ← Use 'audio' not 'document'
                    "format": "mp3",
                    "source": {"bytes": audio_data}
                }
            },
            {"text": "Your prompt"}
        ]
    }]
)
```

**Next Steps:**
1. If audio works: Update the main app to use this format
2. If audio fails: The model may not support audio input
3. Check model capabilities in AWS Bedrock console

In [111]:
# Test audio with S3 location
if audio_data and 's3_key' in locals():
    print("🧪 Testing audio with S3 location...")
    
    try:
        response = bedrock_runtime.converse(
            modelId=MODEL_ID,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "audio": {
                                "format": "mp3",
                                "source": {
                                    "s3Location": {
                                        "uri": f"s3://{BUCKET_NAME}/{s3_key}"
                                    }
                                }
                            }
                        },
                        {
                            "text": "Analyze this music performance."
                        }
                    ]
                }
            ]
        )
        
        analysis = response['output']['message']['content'][0]['text']
        print("✅ SUCCESS with S3 location!")
        print("="*60)
        print(analysis)
        print("="*60)
        
    except Exception as e:
        print(f"❌ S3 location failed: {str(e)}")
        
        # Final conclusion
        print("\n" + "="*60)
        print("📋 CONCLUSION:")
        print("❌ Audio content blocks are NOT supported by this model")
        print("✅ Use AWS Transcribe + Bedrock text analysis instead")
        print("="*60)
else:
    print("❌ Need to upload to S3 first")

🧪 Testing audio with S3 location...
❌ S3 location failed: An error occurred (ValidationException) when calling the Converse operation: ContentBlock object at messages.0.content.0 must set one of the following keys: text, image, toolUse, toolResult, document, video, cachePoint, searchResult.

📋 CONCLUSION:
❌ Audio content blocks are NOT supported by this model
✅ Use AWS Transcribe + Bedrock text analysis instead
